# Essence Wars: Quickstart

This notebook introduces the Essence Wars RL environment - a deterministic, perfect-information card game designed as an RL research testbed.

**Features:**
- Fast simulation: 60K+ steps/second
- Rich action space: 256 discrete actions
- Complex state: 328-dimensional observation
- Built-in baselines: Random, Greedy, MCTS, Alpha-Beta
- Gymnasium v26 & PettingZoo compatible

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/essence-wars/blob/main/python/notebooks/01_quickstart.ipynb)

## Installation

Install essence-wars from PyPI. On Colab, this may take a minute to compile the Rust bindings.

In [ ]:
# Install essence-wars (skip if already installed locally)
# !pip install essence-wars

# For development installation from source:
# !pip install -e /path/to/essence-wars/python[all]

## Basic Game Interface

The core game engine is exposed through `PyGame` - a lightweight Python wrapper around the high-performance Rust implementation.

In [ ]:
from essence_wars import PyGame
import numpy as np

# Create a game with default decks
game = PyGame()

# Reset with a seed for reproducibility
game.reset(seed=42)

# Examine the state
obs = game.observe()
mask = game.action_mask()

print(f"Observation shape: {obs.shape}")  # (328,)
print(f"Action mask shape: {mask.shape}")  # (256,)
print(f"Valid actions: {np.sum(mask > 0)} / {len(mask)}")
print(f"Current player: {game.current_player()}")
print(f"Turn number: {game.turn_number()}")

## Available Decks

Essence Wars features three factions (Argentum, Symbiote, Obsidion) with multiple deck archetypes each.

In [ ]:
# List all available decks
decks = PyGame.list_decks()
print(f"Available decks ({len(decks)}):")
for deck in sorted(decks):
    print(f"  - {deck}")

## Playing a Game with Built-in Bots

The engine includes several built-in opponents for testing and baseline comparison.

In [ ]:
# Create a game between two different decks
game = PyGame(deck1='artificer_tokens', deck2='broodmother_pack')
game.reset(seed=123)

# Play a game using the greedy heuristic bot
steps = 0
while not game.is_done():
    # Use greedy_action for a simple heuristic-based move
    action = game.greedy_action()
    reward, done = game.step(action)
    steps += 1

print(f"Game finished in {steps} steps")
result = game.get_reward(0)  # +1 = P1 wins, -1 = P1 loses, 0 = draw
print(f"Result: {'Player 1 wins!' if result > 0 else 'Player 2 wins!' if result < 0 else 'Draw'}")
print(f"Final reward (P1 perspective): {game.get_reward(0)}")

### Different Bot Types

In [ ]:
# Available bot methods:
game.reset(seed=42)

# 1. Random - uniformly random valid action
random_action = game.random_action()
print(f"Random action: {random_action}")

# 2. Greedy - heuristic-based (fast, decent play)
greedy_action = game.greedy_action()
print(f"Greedy action: {greedy_action}")

# 3. MCTS - Monte Carlo Tree Search (stronger, slower)
mcts_action = game.mcts_action(simulations=100)
print(f"MCTS action (100 sims): {mcts_action}")

# 4. Alpha-Beta - minimax with pruning (deterministic)
ab_action = game.alphabeta_action(depth=4)
print(f"Alpha-Beta action (depth 4): {ab_action}")

## Gymnasium Environment

For RL training, use the `EssenceWarsEnv` class which follows the Gymnasium API.

In [ ]:
from essence_wars import EssenceWarsEnv
import numpy as np

# Create environment (plays against greedy opponent by default)
env = EssenceWarsEnv(
    deck1='artificer_tokens',  # Your deck
    deck2='broodmother_pack',  # Opponent deck
    opponent='greedy',         # Built-in opponent
)

# Standard Gymnasium loop
obs, info = env.reset(seed=42)
total_reward = 0
steps = 0

while True:
    # Get valid actions from action mask
    mask = info['action_mask']
    valid_actions = np.where(mask > 0)[0]
    
    # Random policy (replace with your agent)
    action = np.random.choice(valid_actions)
    
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    
    if terminated or truncated:
        break

print(f"Episode finished in {steps} steps")
print(f"Total reward: {total_reward}")
print(f"Result: {'Won!' if total_reward > 0 else 'Lost' if total_reward < 0 else 'Draw'}")

env.close()

## Observation Space

The 328-dimensional observation encodes the complete game state.

In [ ]:
env = EssenceWarsEnv()
obs, info = env.reset(seed=42)

print("Observation Space Layout:")
print("-" * 50)
print(f"  [0-15]     Global state (turn, phase, essence, AP)")
print(f"  [16-75]    Player 1 creatures (5 slots x 12 features)")
print(f"  [76-85]    Player 1 supports (2 slots x 5 features)")
print(f"  [86-155]   Player 1 hand (7 cards x 10 features)")
print(f"  [156-225]  Player 2 creatures + supports")
print(f"  [226-232]  Player 2 hand count (hidden info)")
print(f"  [233-327]  Commander abilities")
print("-" * 50)
print(f"Total: {len(obs)} floats")

# Sample some values
print(f"\nSample values:")
print(f"  Turn number: {obs[0]:.0f}")
print(f"  Current phase: {obs[1]:.0f}")
print(f"  P1 Essence: {obs[4] * 10:.0f}")
print(f"  P1 Life: {obs[6] * 20:.0f}")

env.close()

## Action Space

The 256-dimensional action space covers all possible game actions.

In [ ]:
env = EssenceWarsEnv()
obs, info = env.reset(seed=42)

print("Action Space Layout:")
print("-" * 50)
print(f"  [0-34]     Play card from hand to slot")
print(f"  [35-59]    Attack with creature")
print(f"  [60-84]    Use creature ability")
print(f"  [255]      End turn")
print("-" * 50)

# Show currently valid actions
mask = info['action_mask']
valid = np.where(mask > 0)[0]
print(f"\nCurrently valid actions: {len(valid)}")
print(f"Action indices: {valid[:10]}..." if len(valid) > 10 else f"Action indices: {valid}")

env.close()

## High-Throughput Training with Vectorized Environments

For efficient training, use `VectorizedEssenceWars` which runs many games in parallel.

In [ ]:
from essence_wars import VectorizedEssenceWars
import time

# Create 64 parallel environments
num_envs = 64
vec_env = VectorizedEssenceWars(num_envs=num_envs)
obs, masks = vec_env.reset(seed=42)

print(f"Observations shape: {obs.shape}")   # (64, 328)
print(f"Action masks shape: {masks.shape}") # (64, 256)

# Benchmark throughput
num_steps = 1000
start_time = time.time()

for _ in range(num_steps):
    # Select random valid actions for all envs
    actions = np.array([
        np.random.choice(np.where(masks[i] > 0)[0])
        for i in range(num_envs)
    ], dtype=np.uint8)
    
    obs, rewards, dones, masks = vec_env.step(actions)

elapsed = time.time() - start_time
total_steps = num_steps * num_envs
sps = total_steps / elapsed

print(f"\nThroughput: {sps:,.0f} steps/second")
print(f"({num_steps} batches x {num_envs} envs = {total_steps:,} total steps)")

vec_env.close()

## Next Steps

- **[02_training_ppo.ipynb](02_training_ppo.ipynb)**: Train a PPO agent
- **[03_evaluation.ipynb](03_evaluation.ipynb)**: Evaluate and benchmark agents

For more information:
- [RESEARCHER_QUICKSTART.md](../docs/RESEARCHER_QUICKSTART.md): Full API reference
- [essence-wars-design.md](../../docs/essence-wars-design.md): Game rules and mechanics